<div style="background:#A6192E;color:#fff;padding:22px 28px;border-radius:14px;font-family:'Barlow Condensed','Arial Narrow',Arial,sans-serif">
<div style="letter-spacing:.2em;text-transform:uppercase;font-size:13px;opacity:.9">Python para Finanzas y Análisis de Datos · Sesión 2</div>
<div style="font-size:34px;font-weight:300;text-transform:uppercase;margin-top:6px"><span style="background:#fff;color:#A6192E;border-radius:10px;padding:2px 12px;margin-right:14px;font-weight:500">2.2</span>Screening de empresas</div>
</div>

**Objetivo.** comparar decenas de empresas por sus indicadores y generar un ranking cambiando solo una lista.

**Lo que vamos a hacer**

- Descargar fundamentales de ~60 empresas.
- Filtrar por valor, calidad y deuda.
- Construir un score compuesto con z-scores.
- Elegir las mejores de cada sector.

In [ ]:
import sys, pathlib, subprocess, warnings
warnings.filterwarnings("ignore")

# Localiza la carpeta del curso (en Google Colab clona el repositorio)
root = pathlib.Path.cwd().resolve()
while root != root.parent and not (root / "common").exists():
    root = root.parent
if not (root / "common").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Ulysess824/Python-for-Finance"], check=False)
    root = pathlib.Path("Python-for-Finance/python-finance-course").resolve()
sys.path.insert(0, str(root))
# En Colab, ejecuta antes:  !pip install -q yfinance arch

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import *
set_plot_style()

## 1. Universo

In [ ]:
tickers = ["AAPL","MSFT","NVDA","GOOGL","AMZN","META","TSLA","AVGO","ORCL","CRM",
           "JPM","BAC","WFC","GS","MS","C","AXP","BLK","SCHW","USB",
           "XOM","CVX","COP","SLB","EOG","EC",
           "JNJ","PFE","MRK","ABBV","LLY","UNH","TMO","ABT",
           "KO","PEP","PG","WMT","COST","MCD","NKE","SBUX","HD","LOW",
           "CAT","BA","HON","GE","UPS","LMT","MMM",
           "T","VZ","DIS","NFLX","CMCSA","CIB","BCH"]
fund = get_fundamentals(tickers)          # con conexión puede tardar 1-2 minutos
fund["sector"] = fund["sector"].fillna("Sin sector")
print(f"{len(fund)} empresas")
display(fund.describe().T[["count", "mean", "50%", "min", "max"]])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
for a, col in zip(ax, ["pe", "roe", "debt_to_equity"]):
    fund[col].dropna().clip(fund[col].quantile(.02), fund[col].quantile(.98)).hist(ax=a, bins=20, color="#d9d9d9", edgecolor="white")
    a.set_title(col)
plt.tight_layout(); plt.show()

## 2. Filtros

> ### ✏️ Ejercicio 1 — Screening clásico
> Quédate con empresas con P/E < 25, ROE > 12% y deuda/patrimonio < 150. Guarda en `screen`.

In [ ]:
# TODO: screen = ...
screen = None

In [ ]:
screen = fund[(fund.pe < 25) & (fund.roe > 0.12) & (fund.debt_to_equity < 150)]
print(f"{len(screen)} de {len(fund)} empresas pasan el filtro")
display(screen[["sector", "pe", "roe", "debt_to_equity"]].round(3))

## 3. Score compuesto

Un filtro es binario. Un **score** ordena: combinamos valor (P/E bajo), calidad (ROE alto) y rentabilidad (margen alto) en z-scores.

> ### ✏️ Ejercicio 2 — Ranking
> Crea `ranking` con columna `score` = promedio de `-z(pe)`, `z(roe)` y `z(profit_margin)`, ordenado de mayor a menor. Limita `pe` entre 0 y 60 antes.

In [ ]:
def z(s):
    return (s - s.mean()) / s.std()
# TODO: ranking = ...
ranking = None

In [ ]:
def z(s):
    return (s - s.mean()) / s.std()

d = fund.dropna(subset=["pe", "roe", "profit_margin"]).copy()
d["pe"] = d["pe"].clip(0, 60)
d["score"] = (-z(d.pe) + z(d.roe) + z(d.profit_margin)) / 3
ranking = d.sort_values("score", ascending=False)
display(ranking[["sector", "pe", "roe", "profit_margin", "score"]].head(10).round(3))

## 4. Por sector

> ### ✏️ Ejercicio 3 — Las 2 mejores de cada sector
> Usa `groupby` para obtener `top_sector`.

In [ ]:
# TODO: top_sector = ...
top_sector = None

In [ ]:
top_sector = ranking.groupby("sector").head(2).sort_values(["sector", "score"], ascending=[True, False])
display(top_sector[["sector", "pe", "roe", "score"]].round(3))

In [ ]:
top = ranking.head(10)[::-1]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(top.index, top.score, color="#A6192E"); ax.set_title("Top 10 por score"); plt.show()

## Para llevarse

- Para cambiar de universo basta editar la lista `tickers`; en Excel serían hojas nuevas.
- Un score da un orden; los filtros duros solo aprueban o descartan.
- Los fundamentales de Yahoo pueden tener huecos: siempre revisa `count` antes de confiar.

---
*Excel para mirar, Python para repetir.*